## Data Loading

In [1]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"


CPU times: total: 2.17 s
Wall time: 2.34 s


In [2]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,3321,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,1810,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,1332,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,432,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,442,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,264,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,281,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,70,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,64,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3


#  RF and XGBoost (Extreme Gradient Boosting)
Hard‐Voting Ensemble model



## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [15]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


Train each base model with sample weights

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)

from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Train XGBoost with sample weights
xgb_strat_y = XGBClassifier(
     objective="multi:softprob",
     num_class=len(algorithms),      # = 5 in your case
     n_estimators=400,
     max_depth=10,
     learning_rate=0.1,              # you can tune this later
     subsample=0.8,
     use_label_encoder=False,
     random_state=42
 )

xgb_strat_y.fit(X_train, y_train, sample_weight=weights_train)



XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_class=5, ...)

Build a hard‐voting ensemble and fitting it

In [17]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y = VotingClassifier(
    estimators=[('rf', rf_strat_y), ('xgb', xgb_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=400)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.1, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=10,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=400, n_jobs=None,
                                            num_class=5, ...))])

Classification report and Feature Importances

In [18]:
# 4) Predict & evaluate
# y_pred_ens = ensemble.predict(X_test)
# print("Ensemble Classification Report:")
# print(classification_report(y_test, y_pred_ens, sample_weight=weights_test))


# Reversing mapping from integer labels to algorithm names
y_pred_int = ensemble_strat_y.predict(X_test)
y_pred_str = le.inverse_transform(y_pred_int)


# Evaluate
print("Classification Report:")
print(classification_report(y_test_str, y_pred_str, sample_weight=weights_test))


Classification Report:
              precision    recall  f1-score   support

       AltRB       0.46      0.53      0.49 1640.270549672929
    Pivot_Br       0.42      0.57      0.48 870.9895672609558
        S_Br       0.85      0.91      0.88 2890.803767280769
        UMKP       0.02      0.00      0.01 1409.7178679964375
        kPEX       0.86      0.89      0.87 8862.24533427892

    accuracy                           0.76 15674.027086490012
   macro avg       0.52      0.58      0.55 15674.027086490012
weighted avg       0.71      0.76      0.73 15674.027086490012



#### Inference on 20% dataset

In [19]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_XGB/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_XGB/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 80.75%

Precision by Algorithm:
  UMKP: 77.78% (9 predictions)
  AltRB: 74.19% (31 predictions)
  kPEX: 84.72% (72 predictions)
  Pivot_Br: 76.36% (55 predictions)
  S_Br: 84.78% (46 predictions)

Recall by Algorithm:
  UMKP: 73.68% (28/38 cases)
  AltRB: 92.50% (37/40 cases)
  kPEX: 91.27% (115/126 cases)
  Pivot_Br: 92.93% (92/99 cases)
  S_Br: 90.10% (91/101 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 25562.60
  Total Optimal Time   = 20524.50
  Time Ratio (pred/opt)= 1.245
  Avg Overhead/case ((pred - opt) / opt)    = 65.46%


Saving full predictions to './ensemble_results/RF_XGB/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 41 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                 

#### Re-evalute on entire dataset

In [20]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_XGB/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_XGB/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 93.82%

Precision by Algorithm:
  UMKP: 96.92% (65 predictions)
  AltRB: 92.08% (101 predictions)
  kPEX: 95.44% (241 predictions)
  Pivot_Br: 90.22% (184 predictions)
  S_Br: 95.27% (169 predictions)

Recall by Algorithm:
  UMKP: 90.00% (90/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 95.87% (348/363 cases)
  Pivot_Br: 96.90% (219/226 cases)
  S_Br: 95.70% (245/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 71932.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.076
  Avg Overhead/case ((pred - opt) / opt)    = 18.38%


Saving full predictions to './ensemble_results/RF_XGB/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 47 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on both Y and k**

In [21]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [22]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

# Train XGBoost with sample weights
xgb_strat_y_K = XGBClassifier(
     objective="multi:softprob",
     num_class=len(algorithms),      # = 5 in your case
     n_estimators=400,
     max_depth=10,
     learning_rate=0.1,              # you can tune this later
     subsample=0.8,
     eval_metric="mlogloss",
     random_state=42
 )

xgb_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None, num_class=5, ...)

Build a soft‐voting ensemble and fitting it

In [23]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_strat_y_K), ('xgb', xgb_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=10,
                                                     n_estimators=400)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric='mlogloss',
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.1, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=10,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=400, n_jobs=None,
                                            num_class=5, ...))])

#### Inference on 20% dataset

In [24]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_XGB/',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_XGB//model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 76.06%

Precision by Algorithm:
  UMKP: 50.00% (6 predictions)
  AltRB: 63.33% (30 predictions)
  kPEX: 75.76% (66 predictions)
  Pivot_Br: 76.12% (67 predictions)
  S_Br: 88.64% (44 predictions)

Recall by Algorithm:
  UMKP: 61.11% (22/36 cases)
  AltRB: 87.80% (36/41 cases)
  kPEX: 89.43% (110/123 cases)
  Pivot_Br: 89.90% (89/99 cases)
  S_Br: 84.47% (87/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 13523.10
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.091
  Avg Overhead/case ((pred - opt) / opt)    = 6.58%


Saving full predictions to './ensemble_results/RF_XGB/all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 51 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset 

#### Re-evalute on entire dataset

In [25]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_XGB/',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_XGB/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 92.37%

Precision by Algorithm:
  UMKP: 94.92% (59 predictions)
  AltRB: 89.42% (104 predictions)
  kPEX: 93.85% (260 predictions)
  Pivot_Br: 87.35% (166 predictions)
  S_Br: 95.91% (171 predictions)

Recall by Algorithm:
  UMKP: 87.00% (87/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 94.77% (344/363 cases)
  Pivot_Br: 95.58% (216/226 cases)
  S_Br: 93.36% (239/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 68023.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.017
  Avg Overhead/case ((pred - opt) / opt)    = 1.83%


Saving full predictions to './ensemble_results/RF_XGB/all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 58 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                   

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [26]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
425,hamming6-2,6,64,1824,28.5,57,57,57,1,64.0,64,64,0.0,0.0,19.9,0.0,6.7,6.8,11.2


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [27]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [28]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [29]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [30]:
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42 
)

rf_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)


# Train XGBoost with sample weights
xgb_holdout_strat_y = XGBClassifier(
     objective="multi:softprob",
     num_class=len(data_splits.algorithms),      # = 5 in your case
     n_estimators=200,
     max_depth=10,
     learning_rate=0.1,              # you can tune this later
     use_label_encoder=False,
    #  random_state=42
 )

xgb_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_class=5, ...)

In [31]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y), ('xgb', xgb_holdout_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=200)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.1, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=10,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=200, n_jobs=None,
                                            num_class=5, ...))])

#### Inference on 20% dataset

In [32]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 75.12%

Precision by Algorithm:
  UMKP: 57.14% (7 predictions)
  AltRB: 64.52% (31 predictions)
  kPEX: 74.55% (55 predictions)
  Pivot_Br: 75.71% (70 predictions)
  S_Br: 84.00% (50 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 90.24% (37/41 cases)
  kPEX: 86.18% (106/123 cases)
  Pivot_Br: 90.91% (90/99 cases)
  S_Br: 86.41% (89/103 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 15315.30
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.236
  Avg Overhead/case ((pred - opt) / opt)    = 3225.12%


Saving full predictions to './ensemble_results/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 53 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset  k_value      weight predicted  \
684         

#### Re-evalute on entire dataset

In [33]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 92.63%

Precision by Algorithm:
  UMKP: 95.31% (64 predictions)
  AltRB: 89.32% (103 predictions)
  kPEX: 93.94% (231 predictions)
  Pivot_Br: 90.00% (190 predictions)
  S_Br: 94.77% (172 predictions)

Recall by Algorithm:
  UMKP: 89.00% (89/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 94.77% (344/363 cases)
  Pivot_Br: 96.02% (217/226 cases)
  S_Br: 94.53% (242/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 69808.40
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.044
  Avg Overhead/case ((pred - opt) / opt)    = 9.14%


Saving full predictions to './ensemble_results/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 56 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
              

### **Balanced train-test split based on Y and K**

In [34]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [35]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [36]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42  best
)

rf_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

from xgboost import XGBClassifier

# Train XGBoost with sample weights
xgb_holdout_strat_y_K = XGBClassifier(
     objective="multi:softprob",
     num_class=len(data_splits.algorithms),      # = 5 in your case
     n_estimators=150,
     max_depth=15,
     learning_rate=0.75,              # you can tune this later
     use_label_encoder=False,
     random_state=42
 )

xgb_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.75, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=15,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=150,
              n_jobs=None, num_class=5, ...)

In [37]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y_K), ('xgb', xgb_holdout_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=18,
                                                     n_estimators=400)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.75, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=15,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=150, n_jobs=None,
                                            num_class=5, ...))])

#### Inference on 20% dataset

In [38]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 76.06%

Precision by Algorithm:
  UMKP: 66.67% (9 predictions)
  AltRB: 65.62% (32 predictions)
  kPEX: 77.97% (59 predictions)
  Pivot_Br: 73.77% (61 predictions)
  S_Br: 84.62% (52 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 90.24% (37/41 cases)
  kPEX: 86.18% (106/123 cases)
  Pivot_Br: 89.90% (89/99 cases)
  S_Br: 89.32% (92/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 17061.20
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.377
  Avg Overhead/case ((pred - opt) / opt)    = 53.51%


Saving full predictions to './ensemble_results/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 51 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         

#### Re-evalute on entire dataset

In [39]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 92.89%

Precision by Algorithm:
  UMKP: 95.31% (64 predictions)
  AltRB: 89.62% (106 predictions)
  kPEX: 94.35% (230 predictions)
  Pivot_Br: 90.40% (177 predictions)
  S_Br: 94.54% (183 predictions)

Recall by Algorithm:
  UMKP: 89.00% (89/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 94.77% (344/363 cases)
  Pivot_Br: 95.13% (215/226 cases)
  S_Br: 96.09% (246/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 71573.60
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.070
  Avg Overhead/case ((pred - opt) / opt)    = 14.94%


Saving full predictions to './ensemble_results/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 54 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
           

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

# ----------------------- ##

#  RF and SVC (RBF kernel)
Hard‐Voting Ensemble model



## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [40]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


Train each base model with sample weights

In [41]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)

from sklearn.svm import SVC
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Instantiate & train SVC with RBF kernel
svc_strat_y = SVC(kernel='rbf', C=0.1, gamma=0.01, probability=True, random_state=42)

svc_strat_y.fit(X_train_scaled, y_train, sample_weight=weights_train)




SVC(C=0.1, gamma=0.01, probability=True, random_state=42)

Build a hard‐voting ensemble and fitting it

In [42]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y = VotingClassifier(
    estimators=[('rf', rf_strat_y), ('svc_rbf', svc_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=400)),
                             ('svc_rbf',
                              SVC(C=0.1, gamma=0.01, probability=True,
                                  random_state=42))])

Classification report and Feature Importances

In [43]:
# 4) Predict & evaluate
# y_pred_ens = ensemble.predict(X_test)
# print("Ensemble Classification Report:")
# print(classification_report(y_test, y_pred_ens, sample_weight=weights_test))


# Reversing mapping from integer labels to algorithm names
y_pred_int = ensemble_strat_y.predict(X_test)
y_pred_str = le.inverse_transform(y_pred_int)


# Evaluate
print("Classification Report:")
print(classification_report(y_test_str, y_pred_str, sample_weight=weights_test))


Classification Report:
              precision    recall  f1-score   support

       AltRB       0.46      0.52      0.49 1640.270549672929
    Pivot_Br       0.42      0.53      0.47 870.9895672609558
        S_Br       0.84      0.86      0.85 2890.803767280769
        UMKP       0.03      0.01      0.01 1409.7178679964375
        kPEX       0.84      0.90      0.87 8862.24533427892

    accuracy                           0.75 15674.027086490012
   macro avg       0.52      0.56      0.54 15674.027086490012
weighted avg       0.70      0.75      0.72 15674.027086490012



#### Inference on 20% dataset

In [44]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_rbf/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_rbf/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 79.34%

Precision by Algorithm:
  UMKP: 71.43% (14 predictions)
  AltRB: 69.57% (23 predictions)
  kPEX: 79.75% (79 predictions)
  Pivot_Br: 80.39% (51 predictions)
  S_Br: 84.78% (46 predictions)

Recall by Algorithm:
  UMKP: 73.68% (28/38 cases)
  AltRB: 85.00% (34/40 cases)
  kPEX: 92.86% (117/126 cases)
  Pivot_Br: 89.90% (89/99 cases)
  S_Br: 88.12% (89/101 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 25693.60
  Total Optimal Time   = 20524.50
  Time Ratio (pred/opt)= 1.252
  Avg Overhead/case ((pred - opt) / opt)    = 69.95%


Saving full predictions to './ensemble_results/RF_SVC_rbf/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 44 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
            

#### Re-evalute on entire dataset

In [45]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_rbf/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_rbf/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 92.50%

Precision by Algorithm:
  UMKP: 93.94% (66 predictions)
  AltRB: 87.88% (99 predictions)
  kPEX: 92.94% (255 predictions)
  Pivot_Br: 91.28% (172 predictions)
  S_Br: 95.24% (168 predictions)

Recall by Algorithm:
  UMKP: 85.00% (85/100 cases)
  AltRB: 93.97% (109/116 cases)
  kPEX: 96.42% (350/363 cases)
  Pivot_Br: 95.13% (215/226 cases)
  S_Br: 94.53% (242/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 72166.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.079
  Avg Overhead/case ((pred - opt) / opt)    = 19.71%


Saving full predictions to './ensemble_results/RF_SVC_rbf/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 57 wrong predictions with time metrics

Top 5 heaviest wrong predictions:


The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on both Y and k**

In [46]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [47]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

from sklearn.svm import SVC
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Instantiate & train SVC with RBF kernel
svc_strat_y_K = SVC(kernel='rbf', C=0.1, gamma='auto', probability=True, random_state=42)

svc_strat_y_K.fit(X_train_scaled, y_train, sample_weight=weights_train)



SVC(C=0.1, gamma='auto', probability=True, random_state=42)

Build a soft‐voting ensemble and fitting it

In [48]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_strat_y_K), ('svc_rbf', svc_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=10,
                                                     n_estimators=400)),
                             ('svc_rbf',
                              SVC(C=0.1, gamma='auto', probability=True,
                                  random_state=42))])

#### Inference on 20% dataset

In [49]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_rbf/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_rbf/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 76.06%

Precision by Algorithm:
  UMKP: 54.55% (11 predictions)
  AltRB: 68.18% (22 predictions)
  kPEX: 73.42% (79 predictions)
  Pivot_Br: 75.93% (54 predictions)
  S_Br: 89.36% (47 predictions)

Recall by Algorithm:
  UMKP: 63.89% (23/36 cases)
  AltRB: 78.05% (32/41 cases)
  kPEX: 91.87% (113/123 cases)
  Pivot_Br: 87.88% (87/99 cases)
  S_Br: 82.52% (85/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 12643.80
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.021
  Avg Overhead/case ((pred - opt) / opt)    = 10.01%


Saving full predictions to './ensemble_results/RF_SVC_rbf/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 51 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
          

#### Re-evalute on entire dataset

In [50]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_rbf/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_rbf/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 89.34%

Precision by Algorithm:
  UMKP: 90.00% (60 predictions)
  AltRB: 88.30% (94 predictions)
  kPEX: 88.81% (295 predictions)
  Pivot_Br: 85.71% (140 predictions)
  S_Br: 93.57% (171 predictions)

Recall by Algorithm:
  UMKP: 79.00% (79/100 cases)
  AltRB: 88.79% (103/116 cases)
  kPEX: 94.77% (344/363 cases)
  Pivot_Br: 92.04% (208/226 cases)
  S_Br: 89.84% (230/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 67278.60
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.006
  Avg Overhead/case ((pred - opt) / opt)    = 13.67%


Saving full predictions to './ensemble_results/RF_SVC_rbf/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 81 wrong predictions with time metrics

Top 5 heaviest wrong predictions

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [51]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
425,hamming6-2,6,64,1824,28.5,57,57,57,1,64.0,64,64,0.0,0.0,19.9,0.0,6.7,6.8,11.2


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [52]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [53]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [54]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [55]:
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42 
)

rf_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate & train SVC with RBF kernel
svc_holdout_strat_y = SVC(kernel='rbf', C=0.1, gamma='auto', probability=True, random_state=42)

svc_holdout_strat_y.fit(X_train_scaled, y_train_, sample_weight=weights_train_)


SVC(C=0.1, gamma='auto', probability=True, random_state=42)

In [56]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y), ('svc_rbf', svc_holdout_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=200)),
                             ('svc_rbf',
                              SVC(C=0.1, gamma='auto', probability=True,
                                  random_state=42))])

#### Inference on 20% dataset

In [57]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_rbf/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_rbf/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 76.53%

Precision by Algorithm:
  UMKP: 66.67% (12 predictions)
  AltRB: 66.67% (27 predictions)
  kPEX: 75.38% (65 predictions)
  Pivot_Br: 79.66% (59 predictions)
  S_Br: 82.00% (50 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 85.37% (35/41 cases)
  kPEX: 89.43% (110/123 cases)
  Pivot_Br: 88.89% (88/99 cases)
  S_Br: 88.35% (91/103 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 15371.30
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.241
  Avg Overhead/case ((pred - opt) / opt)    = 3222.96%


Saving full predictions to './ensemble_results/RF_SVC_rbf/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 50 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset  k_value      weight predicted  \


#### Re-evalute on entire dataset

In [58]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_rbf/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_rbf/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 91.97%

Precision by Algorithm:
  UMKP: 94.03% (67 predictions)
  AltRB: 87.13% (101 predictions)
  kPEX: 93.09% (246 predictions)
  Pivot_Br: 90.86% (175 predictions)
  S_Br: 93.57% (171 predictions)

Recall by Algorithm:
  UMKP: 85.00% (85/100 cases)
  AltRB: 94.83% (110/116 cases)
  kPEX: 95.04% (345/363 cases)
  Pivot_Br: 95.13% (215/226 cases)
  S_Br: 94.53% (242/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 69974.40
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.046
  Avg Overhead/case ((pred - opt) / opt)    = 8.76%


Saving full predictions to './ensemble_results/RF_SVC_rbf/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 61 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

### **Balanced train-test split based on Y and K**

In [59]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [60]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [61]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42  best
)

rf_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate & train SVC with RBF kernel
svc_holdout_strat_y_K = SVC(kernel='rbf', C=0.1, gamma='auto', probability=True, random_state=42)

svc_holdout_strat_y_K.fit(X_train_scaled, y_train_, sample_weight=weights_train_)


SVC(C=0.1, gamma='auto', probability=True, random_state=42)

In [62]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y_K), ('xgb', svc_holdout_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=18,
                                                     n_estimators=400)),
                             ('xgb',
                              SVC(C=0.1, gamma='auto', probability=True,
                                  random_state=42))])

#### Inference on 20% dataset

In [63]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_rbf/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_rbf/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 77.46%

Precision by Algorithm:
  UMKP: 70.59% (17 predictions)
  AltRB: 72.00% (25 predictions)
  kPEX: 75.38% (65 predictions)
  Pivot_Br: 76.67% (60 predictions)
  S_Br: 86.96% (46 predictions)

Recall by Algorithm:
  UMKP: 72.22% (26/36 cases)
  AltRB: 85.37% (35/41 cases)
  kPEX: 89.43% (110/123 cases)
  Pivot_Br: 89.90% (89/99 cases)
  S_Br: 88.35% (91/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 12643.10
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.020
  Avg Overhead/case ((pred - opt) / opt)    = 9.39%


Saving full predictions to './ensemble_results/RF_SVC_rbf/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 48 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
              

#### Re-evalute on entire dataset

In [64]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_rbf/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_rbf/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 92.24%

Precision by Algorithm:
  UMKP: 93.24% (74 predictions)
  AltRB: 88.66% (97 predictions)
  kPEX: 93.16% (234 predictions)
  Pivot_Br: 90.06% (181 predictions)
  S_Br: 94.83% (174 predictions)

Recall by Algorithm:
  UMKP: 86.00% (86/100 cases)
  AltRB: 94.83% (110/116 cases)
  kPEX: 94.77% (344/363 cases)
  Pivot_Br: 95.58% (216/226 cases)
  S_Br: 94.92% (243/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 67246.00
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.005
  Avg Overhead/case ((pred - opt) / opt)    = 2.83%


Saving full predictions to './ensemble_results/RF_SVC_rbf/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 59 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
  

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

# ------------------ #

#  RF and SVC (Linear kernel)
Hard‐Voting Ensemble model



## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [3]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


Train each base model with sample weights

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)

from sklearn.svm import SVC
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Instantiate & train SVC with RBF kernel
svc_strat_y_linear = SVC(kernel='linear', C=1.0, gamma='auto', probability=True, random_state=42)

svc_strat_y_linear.fit(X_train_scaled, y_train, sample_weight=weights_train)




SVC(gamma='auto', kernel='linear', probability=True, random_state=42)

Build a hard‐voting ensemble and fitting it

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y = VotingClassifier(
    estimators=[('rf', rf_strat_y), ('svc', svc_strat_y_linear)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y.fit(X_train, y_train, sample_weight=weights_train)


Classification report and Feature Importances

In [ ]:
# 4) Predict & evaluate
# y_pred_ens = ensemble.predict(X_test)
# print("Ensemble Classification Report:")
# print(classification_report(y_test, y_pred_ens, sample_weight=weights_test))


# Reversing mapping from integer labels to algorithm names
y_pred_int = ensemble_strat_y.predict(X_test)
y_pred_str = le.inverse_transform(y_pred_int)


# Evaluate
print("Classification Report:")
print(classification_report(y_test_str, y_pred_str, sample_weight=weights_test))


Classification Report:
              precision    recall  f1-score   support

       AltRB       0.46      0.52      0.49 1640.270549672929
    Pivot_Br       0.25      0.23      0.24 870.9895672609558
        S_Br       0.77      0.87      0.81 2890.803767280769
        UMKP       0.03      0.01      0.01 1409.7178679964375
        kPEX       0.84      0.90      0.87 8862.24533427892

    accuracy                           0.73 15674.027086490012
   macro avg       0.47      0.50      0.48 15674.027086490012
weighted avg       0.68      0.73      0.71 15674.027086490012



#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_linear/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_linear/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 79.81%

Precision by Algorithm:
  UMKP: 69.23% (13 predictions)
  AltRB: 69.57% (23 predictions)
  kPEX: 80.77% (78 predictions)
  Pivot_Br: 83.33% (54 predictions)
  S_Br: 82.22% (45 predictions)

Recall by Algorithm:
  UMKP: 73.68% (28/38 cases)
  AltRB: 85.00% (34/40 cases)
  kPEX: 92.86% (117/126 cases)
  Pivot_Br: 89.90% (89/99 cases)
  S_Br: 89.11% (90/101 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 25615.80
  Total Optimal Time   = 20524.50
  Time Ratio (pred/opt)= 1.248
  Avg Overhead/case ((pred - opt) / opt)    = 65.00%


Saving full predictions to './ensemble_results/RF_SVC_rbf/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 43 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
            

KeyError: "['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4'] not in index"

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_linear/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_linear/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 92.63%

Precision by Algorithm:
  UMKP: 94.03% (67 predictions)
  AltRB: 87.76% (98 predictions)
  kPEX: 93.70% (254 predictions)
  Pivot_Br: 91.23% (171 predictions)
  S_Br: 94.71% (170 predictions)

Recall by Algorithm:
  UMKP: 85.00% (85/100 cases)
  AltRB: 93.97% (109/116 cases)
  kPEX: 96.14% (349/363 cases)
  Pivot_Br: 95.58% (216/226 cases)
  S_Br: 94.92% (243/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 72084.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.078
  Avg Overhead/case ((pred - opt) / opt)    = 18.32%


Saving full predictions to './ensemble_results/RF_SVC_rbf/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 56 wrong predictions with time metrics

Top 5 heaviest wrong predictions:


KeyError: "['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4'] not in index"

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on both Y and k**

In [ ]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

from sklearn.svm import SVC
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# Instantiate & train SVC with linear kernel
svc_strat_y_K_linear = SVC(kernel='linear', C=1.0, gamma='auto', probability=True, random_state=42)

svc_strat_y_K_linear.fit(X_train_scaled, y_train, sample_weight=weights_train)



SVC(C=0.1, gamma='auto', probability=True, random_state=42)

Build a soft‐voting ensemble and fitting it

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_strat_y_K), ('svc_rbf', svc_strat_y_K_linear)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=10,
                                                     n_estimators=400)),
                             ('svc_rbf',
                              SVC(C=0.1, gamma='auto', probability=True,
                                  random_state=42))])

#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_linear/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_linear/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 74.65%

Precision by Algorithm:
  UMKP: 55.56% (9 predictions)
  AltRB: 66.67% (24 predictions)
  kPEX: 75.00% (84 predictions)
  Pivot_Br: 69.81% (53 predictions)
  S_Br: 88.37% (43 predictions)

Recall by Algorithm:
  UMKP: 58.33% (21/36 cases)
  AltRB: 78.05% (32/41 cases)
  kPEX: 89.43% (110/123 cases)
  Pivot_Br: 87.88% (87/99 cases)
  S_Br: 79.61% (82/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 12641.70
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.020
  Avg Overhead/case ((pred - opt) / opt)    = 9.69%


Saving full predictions to './ensemble_results/RF_SVC_rbf/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 54 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
            

KeyError: "['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4'] not in index"

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_linear/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_linear/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 89.21%

Precision by Algorithm:
  UMKP: 92.98% (57 predictions)
  AltRB: 87.63% (97 predictions)
  kPEX: 88.67% (300 predictions)
  Pivot_Br: 84.51% (142 predictions)
  S_Br: 93.90% (164 predictions)

Recall by Algorithm:
  UMKP: 77.00% (77/100 cases)
  AltRB: 89.66% (104/116 cases)
  kPEX: 93.94% (341/363 cases)
  Pivot_Br: 92.48% (209/226 cases)
  S_Br: 89.06% (228/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 67263.00
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.006
  Avg Overhead/case ((pred - opt) / opt)    = 13.55%


Saving full predictions to './ensemble_results/RF_SVC_rbf/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 82 wrong predictions with time metrics

Top 5 heaviest wrong predictions

KeyError: "['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4'] not in index"

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [ ]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
425,hamming6-2,6,64,1824,28.5,57,57,57,1,64.0,64,64,0.0,0.0,19.9,0.0,6.7,6.8,11.2


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [ ]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [ ]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [ ]:
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42 
)

rf_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate & train SVC with linear kernel
svc_holdout_strat_y = SVC(kernel='linear', C=1.0, gamma='auto', probability=True, random_state=42)

svc_holdout_strat_y.fit(X_train_scaled, y_train_, sample_weight=weights_train_)


SVC(C=0.1, gamma='auto', probability=True, random_state=42)

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y), ('svc_rbf', svc_holdout_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=200)),
                             ('svc_rbf',
                              SVC(C=0.1, gamma='auto', probability=True,
                                  random_state=42))])

#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_linear/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_linear/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 77.46%

Precision by Algorithm:
  UMKP: 68.75% (16 predictions)
  AltRB: 70.83% (24 predictions)
  kPEX: 72.31% (65 predictions)
  Pivot_Br: 83.93% (56 predictions)
  S_Br: 82.69% (52 predictions)

Recall by Algorithm:
  UMKP: 72.22% (26/36 cases)
  AltRB: 85.37% (35/41 cases)
  kPEX: 90.24% (111/123 cases)
  Pivot_Br: 88.89% (88/99 cases)
  S_Br: 88.35% (91/103 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 14609.30
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.179
  Avg Overhead/case ((pred - opt) / opt)    = 3225.62%


Saving full predictions to './ensemble_results/RF_SVC_rbf/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 48 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset  k_value      weight predicted  \


KeyError: "['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4'] not in index"

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_linear/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_linear/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 92.24%

Precision by Algorithm:
  UMKP: 93.06% (72 predictions)
  AltRB: 88.66% (97 predictions)
  kPEX: 92.53% (241 predictions)
  Pivot_Br: 92.35% (170 predictions)
  S_Br: 93.33% (180 predictions)

Recall by Algorithm:
  UMKP: 86.00% (86/100 cases)
  AltRB: 94.83% (110/116 cases)
  kPEX: 95.04% (345/363 cases)
  Pivot_Br: 95.13% (215/226 cases)
  S_Br: 94.92% (243/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 69205.40
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.035
  Avg Overhead/case ((pred - opt) / opt)    = 9.50%


Saving full predictions to './ensemble_results/RF_SVC_rbf/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 59 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
    

KeyError: "['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4'] not in index"

### **Balanced train-test split based on Y and K**

In [ ]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42  best
)

rf_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate & train SVC with RBF kernel
svc_holdout_strat_y_K = SVC(kernel='linear', C=1.0, gamma='auto', probability=True, random_state=42)

svc_holdout_strat_y_K.fit(X_train_scaled, y_train_, sample_weight=weights_train_)


SVC(C=0.1, gamma='auto', probability=True, random_state=42)

In [163]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y_K), ('xgb', svc_holdout_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=18,
                                                     n_estimators=400)),
                             ('xgb',
                              SVC(C=0.1, gamma='auto', probability=True,
                                  random_state=42))])

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y_K), ('xgb', svc_holdout_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=18,
                                                     n_estimators=400)),
                             ('xgb',
                              SVC(C=0.1, gamma='auto', probability=True,
                                  random_state=42))])

#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_linear/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_linear/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 76.06%

Precision by Algorithm:
  UMKP: 64.29% (14 predictions)
  AltRB: 71.43% (28 predictions)
  kPEX: 75.38% (65 predictions)
  Pivot_Br: 75.41% (61 predictions)
  S_Br: 84.44% (45 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 85.37% (35/41 cases)
  kPEX: 87.80% (108/123 cases)
  Pivot_Br: 89.90% (89/99 cases)
  S_Br: 86.41% (89/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 13608.00
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.098
  Avg Overhead/case ((pred - opt) / opt)    = 10.14%


Saving full predictions to './ensemble_results/RF_SVC_rbf/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 51 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
             

KeyError: "['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4'] not in index"

#### Re-evalute on entire dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_SVC_linear/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_SVC_linear/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 91.97%

Precision by Algorithm:
  UMKP: 92.75% (69 predictions)
  AltRB: 88.24% (102 predictions)
  kPEX: 93.13% (233 predictions)
  Pivot_Br: 89.94% (179 predictions)
  S_Br: 94.35% (177 predictions)

Recall by Algorithm:
  UMKP: 85.00% (85/100 cases)
  AltRB: 94.83% (110/116 cases)
  kPEX: 94.49% (343/363 cases)
  Pivot_Br: 95.58% (216/226 cases)
  S_Br: 94.53% (242/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 68210.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.020
  Avg Overhead/case ((pred - opt) / opt)    = 3.02%


Saving full predictions to './ensemble_results/RF_SVC_rbf/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 61 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
 

KeyError: "['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4'] not in index"

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

# ----------- #

#  RF and LR no penalty

Hard‐Voting Ensemble model



### Model

In [104]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.metrics import classification_report

In [105]:
# --- 1) Soft Cross‐Entropy with Temperature --------------------------

class SoftCrossEntropyLoss(nn.Module):
    """
    Cross‐entropy loss where logits are divided by a temperature T before softmax.
    """
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # scale logits
        scaled = logits / self.temperature
        return F.cross_entropy(scaled, targets)

In [106]:
# --- 2) sklearn‐style Torch Logistic Regression -----------------------

class TorchLogisticClassifier(BaseEstimator, ClassifierMixin):
    """
    A sklearn‐compatible PyTorch logistic regression with L1/L2/Elastic-Net
    and soft‐CE temperature scaling.
    """
    def __init__(
        self,
        penalty: str=None,       # none,'l1', 'l2', or 'elasticnet'
        alpha: float = 1e-3,       # regularization strength
        beta: float = 0.5,     # for elastic-net
        lr: float = 1e-2,
        batch_size: int = 64,
        epochs: int = 50,
        temperature: float = 1.0,
        device: str = None,
        random_state: int = 42
    ):
        self.penalty = penalty
        self.alpha = alpha
        self.beta = beta
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.temperature = temperature
        self.device = device
        self.random_state = random_state

    def fit(self, X, y):
        # Validate inputs
        X, y = check_X_y(X, y)
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        # Device
        self.device_ = torch.device(
            self.device or ('cuda' if torch.cuda.is_available() else 'cpu')
        )
        torch.manual_seed(self.random_state)

        # Scale features
        self.scaler_ = StandardScaler().fit(X)
        X_scaled = self.scaler_.transform(X)

        # Map classes to indices
        self.class_to_idx_ = {c:i for i,c in enumerate(self.classes_)}
        y_idx = np.vectorize(self.class_to_idx_.get)(y)

        # Build DataLoader
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        y_tensor = torch.tensor(y_idx,    dtype=torch.long,   device=self.device_)
        dataset  = TensorDataset(X_tensor, y_tensor)
        loader   = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        # Model
        self.model_ = nn.Linear(n_features, n_classes).to(self.device_)
        criterion = SoftCrossEntropyLoss(self.temperature)
        optimizer = optim.SGD(self.model_.parameters(), lr=self.lr)

        # Training loop
        for epoch in range(1, self.epochs + 1):
            self.model_.train()
            for xb, yb in loader:
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss   = criterion(logits, yb)
                # print('loss before regularization', loss)

                # Regularization
                l1 = sum(p.abs().sum() for p in self.model_.parameters())
                l2 = sum(p.pow(2).sum() for p in self.model_.parameters())
                # print('l1, l2, Loss', l1, l2, loss)
                if self.penalty == 'l1':
                    loss = loss + self.alpha * l1
                    # print('l1 loss', loss, l1)
                elif self.penalty == 'l2':
                    loss = loss + self.alpha * l2
                    # print('l2 loss', loss, l2)
                elif self.penalty == 'elasticnet':
                    loss = loss + self.alpha * (self.beta * l1 + (1 - self.beta) * l2)
                    # print('elasticnet', loss)
                else:
                    loss = loss
                    # print('loss', loss)


                loss.backward()
                optimizer.step()

        return self

    def predict(self, X):
        check_is_fitted(self, ['model_', 'scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            preds  = logits.argmax(dim=1).cpu().numpy()
        idx_to_class = {i:c for c,i in self.class_to_idx_.items()}
        return np.vectorize(idx_to_class.get)(preds)

    def predict_proba(self, X):
        check_is_fitted(self, ['model_', 'scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            probs  = F.softmax(logits / self.temperature, dim=1).cpu().numpy()
        return probs

## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [107]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


Train each base model with sample weights

In [108]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)

# Instantiate and train
clf_strat_y = TorchLogisticClassifier(
    penalty=None, # None, 'l1','l2', or 'elasticnet'
    alpha=0,           # regularization strength
    lr=0.75,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time. 
    batch_size=128, 
    epochs=100,
    temperature=1.0
)
clf_strat_y.fit(X_train, y_train)




TorchLogisticClassifier(alpha=0, batch_size=128, epochs=100, lr=0.75)

Build a hard‐voting ensemble and fitting it

In [82]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y = VotingClassifier(
    estimators=[('rf', rf_strat_y), ('lr0', clf_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=400)),
                             ('lr0',
                              TorchLogisticClassifier(alpha=0, batch_size=128,
                                                      epochs=100, lr=0.75))])

#### Inference on 20% dataset

In [83]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR0/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR0/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 79.34%

Precision by Algorithm:
  UMKP: 92.86% (14 predictions)
  AltRB: 66.67% (36 predictions)
  kPEX: 84.51% (71 predictions)
  Pivot_Br: 80.95% (42 predictions)
  S_Br: 76.00% (50 predictions)

Recall by Algorithm:
  UMKP: 68.42% (26/38 cases)
  AltRB: 92.50% (37/40 cases)
  kPEX: 90.48% (114/126 cases)
  Pivot_Br: 87.88% (87/99 cases)
  S_Br: 90.10% (91/101 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 26055.10
  Total Optimal Time   = 20524.50
  Time Ratio (pred/opt)= 1.269
  Avg Overhead/case ((pred - opt) / opt)    = 93.09%


Saving full predictions to './ensemble_results/RF_LR0/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 44 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                

#### Re-evalute on entire dataset

In [84]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR0/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR0/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 87.76%

Precision by Algorithm:
  UMKP: 93.62% (47 predictions)
  AltRB: 67.88% (137 predictions)
  kPEX: 95.20% (229 predictions)
  Pivot_Br: 92.81% (167 predictions)
  S_Br: 87.22% (180 predictions)

Recall by Algorithm:
  UMKP: 60.00% (60/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 92.29% (335/363 cases)
  Pivot_Br: 93.81% (212/226 cases)
  S_Br: 93.36% (239/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 73156.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.094
  Avg Overhead/case ((pred - opt) / opt)    = 28.96%


Saving full predictions to './ensemble_results/RF_LR0/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 93 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on both Y and k**

In [85]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [86]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

# Instantiate and train
clf_strat_y_K = TorchLogisticClassifier(
    penalty=None, # None, 'l1','l2', or 'elasticnet'
    alpha=0,           # regularization strength
    lr=0.25,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time. 
    batch_size=128, 
    epochs=100,
    temperature=1.0
)
clf_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(alpha=0, batch_size=128, epochs=100, lr=0.25)

Build a soft‐voting ensemble and fitting it

In [87]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_strat_y_K), ('lr0', clf_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=10,
                                                     n_estimators=400)),
                             ('lr0',
                              TorchLogisticClassifier(alpha=0, batch_size=128,
                                                      epochs=100, lr=0.25))])

#### Inference on 20% dataset

In [88]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR0/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR0/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 77.46%

Precision by Algorithm:
  UMKP: 55.56% (9 predictions)
  AltRB: 65.91% (44 predictions)
  kPEX: 79.63% (54 predictions)
  Pivot_Br: 77.59% (58 predictions)
  S_Br: 89.58% (48 predictions)

Recall by Algorithm:
  UMKP: 66.67% (24/36 cases)
  AltRB: 92.68% (38/41 cases)
  kPEX: 86.99% (107/123 cases)
  Pivot_Br: 89.90% (89/99 cases)
  S_Br: 87.38% (90/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 13773.10
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.112
  Avg Overhead/case ((pred - opt) / opt)    = 12.79%


Saving full predictions to './ensemble_results/RF_LR0/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 48 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
               

#### Re-evalute on entire dataset

In [89]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR0/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR0/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 83.68%

Precision by Algorithm:
  UMKP: 83.33% (36 predictions)
  AltRB: 66.44% (146 predictions)
  kPEX: 90.27% (226 predictions)
  Pivot_Br: 86.67% (165 predictions)
  S_Br: 86.63% (187 predictions)

Recall by Algorithm:
  UMKP: 56.00% (56/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 87.88% (319/363 cases)
  Pivot_Br: 89.82% (203/226 cases)
  S_Br: 92.19% (236/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 72539.10
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.085
  Avg Overhead/case ((pred - opt) / opt)    = 13.42%


Saving full predictions to './ensemble_results/RF_LR0/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 124 wrong predictions with time metrics

Top 5 heaviest wrong predictions:


The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [90]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
425,hamming6-2,6,64,1824,28.5,57,57,57,1,64.0,64,64,0.0,0.0,19.9,0.0,6.7,6.8,11.2


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [91]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [92]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [93]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_)



Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [94]:
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42 
)

rf_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate & train SVC with RBF kernel
clf_LR_no_penalty_strat_y = TorchLogisticClassifier(
        penalty=None,
        lr=0.01,
        batch_size=32,
        epochs=1000,
        temperature=1.0
    )

clf_LR_no_penalty_strat_y.fit(X_train_, y_train_)

TorchLogisticClassifier(batch_size=32, epochs=1000)

In [95]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y), ('lr0', clf_LR_no_penalty_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=200)),
                             ('lr0',
                              TorchLogisticClassifier(batch_size=32,
                                                      epochs=1000))])

#### Inference on 20% dataset

In [96]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR0/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR0/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 75.59%

Precision by Algorithm:
  UMKP: 55.56% (9 predictions)
  AltRB: 69.57% (46 predictions)
  kPEX: 78.18% (55 predictions)
  Pivot_Br: 68.52% (54 predictions)
  S_Br: 89.80% (49 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 92.68% (38/41 cases)
  kPEX: 84.55% (104/123 cases)
  Pivot_Br: 88.89% (88/99 cases)
  S_Br: 87.38% (90/103 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 12988.50
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.048
  Avg Overhead/case ((pred - opt) / opt)    = 3221.96%


Saving full predictions to './ensemble_results/RF_LR0/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 52 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset  k_value      weight predicted  \
684  

#### Re-evalute on entire dataset

In [97]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR0/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR0/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 86.58%

Precision by Algorithm:
  UMKP: 86.05% (43 predictions)
  AltRB: 71.43% (140 predictions)
  kPEX: 94.44% (216 predictions)
  Pivot_Br: 85.88% (177 predictions)
  S_Br: 89.67% (184 predictions)

Recall by Algorithm:
  UMKP: 63.00% (63/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 88.98% (323/363 cases)
  Pivot_Br: 93.81% (212/226 cases)
  S_Br: 92.58% (237/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 70725.00
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.057
  Avg Overhead/case ((pred - opt) / opt)    = 18.00%


Saving full predictions to './ensemble_results/RF_LR0/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 102 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
     

### **Balanced train-test split based on Y and K**

In [98]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [99]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")




Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [100]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42  best
)

rf_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate and train
clf_strat_y_K = TorchLogisticClassifier(
    penalty=None, # None, 'l1','l2', or 'elasticnet'
    alpha=0,           # regularization strength
    lr=0.25,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time. 
    batch_size=128, 
    epochs=100,
    temperature=1.0
)
clf_strat_y_K.fit(X_train, y_train)

TorchLogisticClassifier(alpha=0, batch_size=128, epochs=100, lr=0.25)

In [101]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y_K), ('xgb', clf_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=18,
                                                     n_estimators=400)),
                             ('xgb',
                              TorchLogisticClassifier(alpha=0, batch_size=128,
                                                      epochs=100, lr=0.25))])

#### Inference on 20% dataset

In [102]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR0/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR0/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 76.53%

Precision by Algorithm:
  UMKP: 66.67% (12 predictions)
  AltRB: 67.44% (43 predictions)
  kPEX: 76.67% (60 predictions)
  Pivot_Br: 74.07% (54 predictions)
  S_Br: 90.91% (44 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 92.68% (38/41 cases)
  kPEX: 86.99% (107/123 cases)
  Pivot_Br: 88.89% (88/99 cases)
  S_Br: 86.41% (89/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 12812.60
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.034
  Avg Overhead/case ((pred - opt) / opt)    = 12.45%


Saving full predictions to './ensemble_results/RF_LR0/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 50 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                 

#### Re-evalute on entire dataset

In [103]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR0/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR0/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 86.18%

Precision by Algorithm:
  UMKP: 85.00% (40 predictions)
  AltRB: 66.90% (145 predictions)
  kPEX: 93.72% (223 predictions)
  Pivot_Br: 89.88% (168 predictions)
  S_Br: 89.13% (184 predictions)

Recall by Algorithm:
  UMKP: 56.00% (56/100 cases)
  AltRB: 97.41% (113/116 cases)
  kPEX: 90.36% (328/363 cases)
  Pivot_Br: 93.36% (211/226 cases)
  S_Br: 92.58% (237/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 70696.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.057
  Avg Overhead/case ((pred - opt) / opt)    = 13.09%


Saving full predictions to './ensemble_results/RF_LR0/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 105 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

# -------------------- #

#  RF and LR with l1 penalty

Hard‐Voting Ensemble model



## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [109]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


Train each base model with sample weights

In [110]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)

# Instantiate and train
clf_l1_strat_y = TorchLogisticClassifier(
    penalty='l1', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.9,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time. 
    batch_size=128, 
    epochs=100,
    temperature=1.0
)
clf_l1_strat_y.fit(X_train, y_train)




TorchLogisticClassifier(batch_size=128, epochs=100, lr=0.9, penalty='l1')

Build a hard‐voting ensemble and fitting it

In [111]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y = VotingClassifier(
    estimators=[('rf', rf_strat_y), ('lr0', clf_l1_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=400)),
                             ('lr0',
                              TorchLogisticClassifier(batch_size=128,
                                                      epochs=100, lr=0.9,
                                                      penalty='l1'))])

#### Inference on 20% dataset

In [112]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR1/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR1/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 79.34%

Precision by Algorithm:
  UMKP: 86.67% (15 predictions)
  AltRB: 67.57% (37 predictions)
  kPEX: 83.33% (72 predictions)
  Pivot_Br: 79.07% (43 predictions)
  S_Br: 80.43% (46 predictions)

Recall by Algorithm:
  UMKP: 68.42% (26/38 cases)
  AltRB: 95.00% (38/40 cases)
  kPEX: 91.27% (115/126 cases)
  Pivot_Br: 87.88% (87/99 cases)
  S_Br: 88.12% (89/101 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 27520.20
  Total Optimal Time   = 20524.50
  Time Ratio (pred/opt)= 1.341
  Avg Overhead/case ((pred - opt) / opt)    = 181.62%


Saving full predictions to './ensemble_results/RF_LR1/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 44 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
               

#### Re-evalute on entire dataset

In [113]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR0/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR0/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 88.16%

Precision by Algorithm:
  UMKP: 95.65% (46 predictions)
  AltRB: 67.86% (140 predictions)
  kPEX: 94.85% (233 predictions)
  Pivot_Br: 92.07% (164 predictions)
  S_Br: 89.83% (177 predictions)

Recall by Algorithm:
  UMKP: 61.00% (61/100 cases)
  AltRB: 97.41% (113/116 cases)
  kPEX: 93.11% (338/363 cases)
  Pivot_Br: 93.81% (212/226 cases)
  S_Br: 92.58% (237/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 74569.00
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.115
  Avg Overhead/case ((pred - opt) / opt)    = 53.29%


Saving full predictions to './ensemble_results/RF_LR0/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 90 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on both Y and k**

In [114]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [115]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

# Instantiate and train
clf_l1_strat_y_K = TorchLogisticClassifier(
    penalty='l1', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.3,              # Learning rate
    batch_size=128,        
    epochs=100,
    temperature=1.0
)
clf_l1_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=128, epochs=100, lr=0.3, penalty='l1')

Build a soft‐voting ensemble and fitting it

In [116]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_strat_y_K), ('lr0', clf_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=10,
                                                     n_estimators=400)),
                             ('lr0',
                              TorchLogisticClassifier(alpha=0, batch_size=128,
                                                      epochs=100, lr=0.25))])

#### Inference on 20% dataset

In [117]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR1/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR1/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 77.46%

Precision by Algorithm:
  UMKP: 60.00% (10 predictions)
  AltRB: 67.44% (43 predictions)
  kPEX: 79.25% (53 predictions)
  Pivot_Br: 76.27% (59 predictions)
  S_Br: 89.58% (48 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 92.68% (38/41 cases)
  kPEX: 86.18% (106/123 cases)
  Pivot_Br: 89.90% (89/99 cases)
  S_Br: 87.38% (90/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 13773.90
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.112
  Avg Overhead/case ((pred - opt) / opt)    = 12.77%


Saving full predictions to './ensemble_results/RF_LR1/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 48 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
              

#### Re-evalute on entire dataset

In [118]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR1/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR1/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 83.82%

Precision by Algorithm:
  UMKP: 83.78% (37 predictions)
  AltRB: 66.90% (145 predictions)
  kPEX: 90.27% (226 predictions)
  Pivot_Br: 85.80% (169 predictions)
  S_Br: 87.43% (183 predictions)

Recall by Algorithm:
  UMKP: 57.00% (57/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 87.88% (319/363 cases)
  Pivot_Br: 90.27% (204/226 cases)
  S_Br: 91.80% (235/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 71896.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.075
  Avg Overhead/case ((pred - opt) / opt)    = 13.44%


Saving full predictions to './ensemble_results/RF_LR1/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 123 wrong predictions with time metrics

Top 5 heaviest wrong predictions:


The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [119]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
425,hamming6-2,6,64,1824,28.5,57,57,57,1,64.0,64,64,0.0,0.0,19.9,0.0,6.7,6.8,11.2


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [120]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [121]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [122]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")




Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [123]:
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42 
)

rf_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate & train SVC with RBF kernel
clf_l1_strat_y = TorchLogisticClassifier(
        penalty='l1',
        alpha=1e-3,
        lr=1e-2,
        batch_size=128,
        epochs=30,
        temperature=2.0
    )
clf_l1_strat_y.fit(X_train, y_train)

TorchLogisticClassifier(batch_size=128, epochs=30, penalty='l1',
                        temperature=2.0)

In [124]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y), ('lr0', clf_l1_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=200)),
                             ('lr0',
                              TorchLogisticClassifier(batch_size=128, epochs=30,
                                                      penalty='l1',
                                                      temperature=2.0))])

#### Inference on 20% dataset

In [125]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR1/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 74.65%

Precision by Algorithm:
  UMKP: 100.00% (4 predictions)
  AltRB: 67.44% (43 predictions)
  kPEX: 80.43% (46 predictions)
  Pivot_Br: 77.36% (53 predictions)
  S_Br: 71.64% (67 predictions)

Recall by Algorithm:
  UMKP: 63.89% (23/36 cases)
  AltRB: 90.24% (37/41 cases)
  kPEX: 82.93% (102/123 cases)
  Pivot_Br: 82.83% (82/99 cases)
  S_Br: 91.26% (94/103 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 17494.70
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.412
  Avg Overhead/case ((pred - opt) / opt)    = 3485.33%


Saving full predictions to './ensemble_results/RF_LR1/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 54 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset  k_value      weight predicted  \
684 

#### Re-evalute on entire dataset

In [126]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR1/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 81.97%

Precision by Algorithm:
  UMKP: 100.00% (20 predictions)
  AltRB: 71.01% (138 predictions)
  kPEX: 95.31% (192 predictions)
  Pivot_Br: 86.71% (173 predictions)
  S_Br: 72.57% (237 predictions)

Recall by Algorithm:
  UMKP: 47.00% (47/100 cases)
  AltRB: 93.10% (108/116 cases)
  kPEX: 84.02% (305/363 cases)
  Pivot_Br: 89.82% (203/226 cases)
  S_Br: 92.97% (238/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 84979.10
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.271
  Avg Overhead/case ((pred - opt) / opt)    = 1354.60%


Saving full predictions to './ensemble_results/RF_LR1/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 137 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
  

### **Balanced train-test split based on Y and K**

In [127]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [128]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")




Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [129]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42  best
)

rf_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate and train
# Instantiate and train
clf_l1_strat_y_K = TorchLogisticClassifier(
    penalty='l1', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.3,              # Learning rate
    batch_size=128,        
    epochs=100,
    temperature=1.0
)
clf_l1_strat_y_K.fit(X_train, y_train)



TorchLogisticClassifier(batch_size=128, epochs=100, lr=0.3, penalty='l1')

In [131]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y_K), ('xgb', clf_l1_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=18,
                                                     n_estimators=400)),
                             ('xgb',
                              TorchLogisticClassifier(batch_size=128,
                                                      epochs=100, lr=0.3,
                                                      penalty='l1'))])

#### Inference on 20% dataset

In [132]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR1/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 76.06%

Precision by Algorithm:
  UMKP: 63.64% (11 predictions)
  AltRB: 67.44% (43 predictions)
  kPEX: 77.97% (59 predictions)
  Pivot_Br: 73.58% (53 predictions)
  S_Br: 87.23% (47 predictions)

Recall by Algorithm:
  UMKP: 66.67% (24/36 cases)
  AltRB: 92.68% (38/41 cases)
  kPEX: 86.99% (107/123 cases)
  Pivot_Br: 88.89% (88/99 cases)
  S_Br: 86.41% (89/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 13780.40
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.112
  Avg Overhead/case ((pred - opt) / opt)    = 13.37%


Saving full predictions to './ensemble_results/RF_LR1/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 51 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                 

#### Re-evalute on entire dataset

In [133]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR1/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR1/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 85.92%

Precision by Algorithm:
  UMKP: 82.35% (34 predictions)
  AltRB: 66.00% (150 predictions)
  kPEX: 94.12% (221 predictions)
  Pivot_Br: 90.06% (171 predictions)
  S_Br: 89.13% (184 predictions)

Recall by Algorithm:
  UMKP: 52.00% (52/100 cases)
  AltRB: 97.41% (113/116 cases)
  kPEX: 90.91% (330/363 cases)
  Pivot_Br: 93.36% (211/226 cases)
  S_Br: 92.58% (237/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 71670.50
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.072
  Avg Overhead/case ((pred - opt) / opt)    = 13.36%


Saving full predictions to './ensemble_results/RF_LR1/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 107 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

# ------------- #

#  RF and LR with l2 penalty

Hard‐Voting Ensemble model



## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [164]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


Train each base model with sample weights

In [165]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)

clf_l2_strat_y = TorchLogisticClassifier(
    penalty='l2', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.25,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time.
    batch_size=128,       
    epochs=100,
    temperature=1.0
)
clf_l2_strat_y.fit(X_train, y_train)




TorchLogisticClassifier(batch_size=128, epochs=100, lr=0.25, penalty='l2')

Build a hard‐voting ensemble and fitting it

In [166]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y = VotingClassifier(
    estimators=[('rf', rf_strat_y), ('lr0', clf_l2_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=400)),
                             ('lr0',
                              TorchLogisticClassifier(batch_size=128,
                                                      epochs=100, lr=0.25,
                                                      penalty='l2'))])

#### Inference on 20% dataset

In [167]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR2/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 79.81%

Precision by Algorithm:
  UMKP: 92.31% (13 predictions)
  AltRB: 71.43% (35 predictions)
  kPEX: 83.10% (71 predictions)
  Pivot_Br: 80.00% (40 predictions)
  S_Br: 77.78% (54 predictions)

Recall by Algorithm:
  UMKP: 73.68% (28/38 cases)
  AltRB: 92.50% (37/40 cases)
  kPEX: 90.48% (114/126 cases)
  Pivot_Br: 86.87% (86/99 cases)
  S_Br: 90.10% (91/101 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 26049.90
  Total Optimal Time   = 20524.50
  Time Ratio (pred/opt)= 1.269
  Avg Overhead/case ((pred - opt) / opt)    = 93.15%


Saving full predictions to './ensemble_results/RF_LR2/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 43 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                

#### Re-evalute on entire dataset

In [168]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR2/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 88.68%

Precision by Algorithm:
  UMKP: 94.12% (51 predictions)
  AltRB: 71.64% (134 predictions)
  kPEX: 94.81% (231 predictions)
  Pivot_Br: 92.59% (162 predictions)
  S_Br: 88.46% (182 predictions)

Recall by Algorithm:
  UMKP: 68.00% (68/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 92.01% (334/363 cases)
  Pivot_Br: 93.36% (211/226 cases)
  S_Br: 93.36% (239/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 72658.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.086
  Avg Overhead/case ((pred - opt) / opt)    = 28.84%


Saving full predictions to './ensemble_results/RF_LR2/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 86 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on both Y and k**

In [169]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [170]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

# Instantiate and train
clf_l2_strat_y_K = TorchLogisticClassifier(
    penalty='l2', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.25,              # Learning rate
    batch_size=X_train.shape[0],       # Feeding whole data at a time.
    epochs=100,
    temperature=1.0
)
clf_l2_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=848, epochs=100, lr=0.25, penalty='l2')

Build a soft‐voting ensemble and fitting it

In [171]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_strat_y_K), ('lr0', clf_l2_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=10,
                                                     n_estimators=400)),
                             ('lr0',
                              TorchLogisticClassifier(batch_size=848,
                                                      epochs=100, lr=0.25,
                                                      penalty='l2'))])

#### Inference on 20% dataset

In [172]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR2/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 75.59%

Precision by Algorithm:
  UMKP: 54.55% (11 predictions)
  AltRB: 62.50% (40 predictions)
  kPEX: 79.59% (49 predictions)
  Pivot_Br: 77.05% (61 predictions)
  S_Br: 84.62% (52 predictions)

Recall by Algorithm:
  UMKP: 66.67% (24/36 cases)
  AltRB: 87.80% (36/41 cases)
  kPEX: 84.55% (104/123 cases)
  Pivot_Br: 88.89% (88/99 cases)
  S_Br: 86.41% (89/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 13947.70
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.126
  Avg Overhead/case ((pred - opt) / opt)    = 32.37%


Saving full predictions to './ensemble_results/RF_LR2/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 52 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
              

#### Re-evalute on entire dataset

In [173]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR2/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR2/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 83.42%

Precision by Algorithm:
  UMKP: 84.78% (46 predictions)
  AltRB: 69.23% (130 predictions)
  kPEX: 90.41% (219 predictions)
  Pivot_Br: 84.80% (171 predictions)
  S_Br: 83.51% (194 predictions)

Recall by Algorithm:
  UMKP: 63.00% (63/100 cases)
  AltRB: 93.97% (109/116 cases)
  kPEX: 86.23% (313/363 cases)
  Pivot_Br: 89.38% (202/226 cases)
  S_Br: 91.02% (233/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 74104.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.108
  Avg Overhead/case ((pred - opt) / opt)    = 63.62%


Saving full predictions to './ensemble_results/RF_LR2/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 126 wrong predictions with time metrics

Top 5 heaviest wrong predictions:


The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [174]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
425,hamming6-2,6,64,1824,28.5,57,57,57,1,64.0,64,64,0.0,0.0,19.9,0.0,6.7,6.8,11.2


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [175]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [176]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [177]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")




Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [178]:
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42 
)

rf_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

clf_l2_strat_y = TorchLogisticClassifier(
        penalty='l2',
        alpha=1e-2,
        beta=0.7,
        lr=1e-2,
        batch_size=128,
        epochs=30,
        temperature=2.0
    )
clf_l2_strat_y.fit(X_train, y_train)

TorchLogisticClassifier(alpha=0.01, batch_size=128, beta=0.7, epochs=30,
                        penalty='l2', temperature=2.0)

In [179]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y), ('lr02', clf_l2_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=200)),
                             ('lr02',
                              TorchLogisticClassifier(alpha=0.01,
                                                      batch_size=128, beta=0.7,
                                                      epochs=30, penalty='l2',
                                                      temperature=2.0))])

#### Inference on 20% dataset

In [180]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR2/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 74.65%

Precision by Algorithm:
  UMKP: 100.00% (4 predictions)
  AltRB: 65.00% (40 predictions)
  kPEX: 82.22% (45 predictions)
  Pivot_Br: 77.78% (54 predictions)
  S_Br: 71.43% (70 predictions)

Recall by Algorithm:
  UMKP: 63.89% (23/36 cases)
  AltRB: 87.80% (36/41 cases)
  kPEX: 82.93% (102/123 cases)
  Pivot_Br: 83.84% (83/99 cases)
  S_Br: 91.26% (94/103 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 18466.20
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.490
  Avg Overhead/case ((pred - opt) / opt)    = 3487.64%


Saving full predictions to './ensemble_results/RF_LR2/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 54 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset  k_value      weight predicted  \
684 

#### Re-evalute on entire dataset

In [181]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR2/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 82.50%

Precision by Algorithm:
  UMKP: 100.00% (22 predictions)
  AltRB: 70.15% (134 predictions)
  kPEX: 95.88% (194 predictions)
  Pivot_Br: 86.86% (175 predictions)
  S_Br: 73.62% (235 predictions)

Recall by Algorithm:
  UMKP: 49.00% (49/100 cases)
  AltRB: 93.10% (108/116 cases)
  kPEX: 84.57% (307/363 cases)
  Pivot_Br: 91.15% (206/226 cases)
  S_Br: 92.97% (238/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 84144.20
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.258
  Avg Overhead/case ((pred - opt) / opt)    = 175.35%


Saving full predictions to './ensemble_results/RF_LR2/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 133 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

### **Balanced train-test split based on Y and K**

In [182]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [183]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")




Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42  best
)

rf_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate and train
# Instantiate and train
clf_l2_strat_y_K = TorchLogisticClassifier(
    penalty='l2', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    lr=0.25,              # Learning rate
    batch_size=X_train.shape[0],       # Feeding whole data at a time.
    epochs=100,
    temperature=1.0
)
clf_l2_strat_y_K.fit(X_train, y_train)



TorchLogisticClassifier(batch_size=848, epochs=100, lr=0.25, penalty='l2')

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y_K), ('xgb', clf_elasticnet_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=18,
                                                     n_estimators=400)),
                             ('xgb',
                              TorchLogisticClassifier(batch_size=848,
                                                      epochs=100, lr=0.25,
                                                      penalty='l2'))])

#### Inference on 20% dataset

In [186]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR2/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 76.06%

Precision by Algorithm:
  UMKP: 69.23% (13 predictions)
  AltRB: 67.44% (43 predictions)
  kPEX: 78.33% (60 predictions)
  Pivot_Br: 76.00% (50 predictions)
  S_Br: 82.98% (47 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 90.24% (37/41 cases)
  kPEX: 86.99% (107/123 cases)
  Pivot_Br: 86.87% (86/99 cases)
  S_Br: 85.44% (88/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 13946.40
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.126
  Avg Overhead/case ((pred - opt) / opt)    = 31.94%


Saving full predictions to './ensemble_results/RF_LR2/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 51 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                 

#### Re-evalute on entire dataset

In [187]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR2/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR2/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 86.58%

Precision by Algorithm:
  UMKP: 87.50% (48 predictions)
  AltRB: 71.11% (135 predictions)
  kPEX: 93.98% (216 predictions)
  Pivot_Br: 90.74% (162 predictions)
  S_Br: 85.43% (199 predictions)

Recall by Algorithm:
  UMKP: 63.00% (63/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 89.81% (326/363 cases)
  Pivot_Br: 91.59% (207/226 cases)
  S_Br: 92.19% (236/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 73273.40
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.096
  Avg Overhead/case ((pred - opt) / opt)    = 63.52%


Saving full predictions to './ensemble_results/RF_LR2/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 102 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

#  RF and LR with l1+l2 penalty

Hard‐Voting Ensemble model



#### LR with L1+L2 regularization

$loss + \alpha \times (\beta \times l_1 + (1 - \beta) \times l_2)$

## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [188]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


Train each base model with sample weights

In [189]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)

clf_elasticnet_strat_y = TorchLogisticClassifier(
    penalty='elasticnet', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    beta=0.7,         # regularization strength of L1 in usint elasticnet
    lr=0.25,              # Learning rate
    # batch_size=X_train.shape[0],       # Feeding whole data at a time.     
    batch_size=128,  
    epochs=100,
    temperature=1.0
)
clf_elasticnet_strat_y.fit(X_train, y_train)




TorchLogisticClassifier(batch_size=128, beta=0.7, epochs=100, lr=0.25,
                        penalty='elasticnet')

Build a hard‐voting ensemble and fitting it

In [190]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y = VotingClassifier(
    estimators=[('rf', rf_strat_y), ('lr0', clf_elasticnet_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=400)),
                             ('lr0',
                              TorchLogisticClassifier(batch_size=128, beta=0.7,
                                                      epochs=100, lr=0.25,
                                                      penalty='elasticnet'))])

#### Inference on 20% dataset

In [191]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR_elasticnet/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR_elasticnet/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 79.34%

Precision by Algorithm:
  UMKP: 92.31% (13 predictions)
  AltRB: 71.43% (35 predictions)
  kPEX: 82.61% (69 predictions)
  Pivot_Br: 79.07% (43 predictions)
  S_Br: 77.36% (53 predictions)

Recall by Algorithm:
  UMKP: 73.68% (28/38 cases)
  AltRB: 92.50% (37/40 cases)
  kPEX: 89.68% (113/126 cases)
  Pivot_Br: 87.88% (87/99 cases)
  S_Br: 89.11% (90/101 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 26197.20
  Total Optimal Time   = 20524.50
  Time Ratio (pred/opt)= 1.276
  Avg Overhead/case ((pred - opt) / opt)    = 93.35%


Saving full predictions to './ensemble_results/RF_LR_elasticnet/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 44 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
      

#### Re-evalute on entire dataset

In [192]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR_elasticnet/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR_elasticnet/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 88.42%

Precision by Algorithm:
  UMKP: 94.00% (50 predictions)
  AltRB: 71.85% (135 predictions)
  kPEX: 94.71% (227 predictions)
  Pivot_Br: 92.73% (165 predictions)
  S_Br: 87.43% (183 predictions)

Recall by Algorithm:
  UMKP: 68.00% (68/100 cases)
  AltRB: 96.55% (112/116 cases)
  kPEX: 91.74% (333/363 cases)
  Pivot_Br: 93.36% (211/226 cases)
  S_Br: 92.97% (238/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 72813.10
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.089
  Avg Overhead/case ((pred - opt) / opt)    = 28.90%


Saving full predictions to './ensemble_results/RF_LR_elasticnet/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 88 wrong predictions with time metrics

Top 5 heaviest wrong predic

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on both Y and k**

In [193]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [194]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

# Instantiate and train
clf_elasticnet_strat_y_K = TorchLogisticClassifier(
    penalty='elasticnet', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    beta=0.7,         # regularization strength of L1 in usint elasticnet
    lr=0.75,              # Learning rate
    batch_size=X_train.shape[0],       # Feeding whole data at a time.
    epochs=100,
    temperature=1.0
)
clf_elasticnet_strat_y_K.fit(X_train, y_train)


TorchLogisticClassifier(batch_size=848, beta=0.7, epochs=100, lr=0.75,
                        penalty='elasticnet')

Build a soft‐voting ensemble and fitting it

In [195]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_strat_y_K), ('lr0', clf_elasticnet_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=10,
                                                     n_estimators=400)),
                             ('lr0',
                              TorchLogisticClassifier(batch_size=848, beta=0.7,
                                                      epochs=100, lr=0.75,
                                                      penalty='elasticnet'))])

#### Inference on 20% dataset

In [196]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR_elasticnet/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR_elasticnet/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 77.00%

Precision by Algorithm:
  UMKP: 60.00% (10 predictions)
  AltRB: 69.05% (42 predictions)
  kPEX: 78.43% (51 predictions)
  Pivot_Br: 78.33% (60 predictions)
  S_Br: 84.00% (50 predictions)

Recall by Algorithm:
  UMKP: 66.67% (24/36 cases)
  AltRB: 90.24% (37/41 cases)
  kPEX: 86.18% (106/123 cases)
  Pivot_Br: 89.90% (89/99 cases)
  S_Br: 87.38% (90/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 13948.50
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.126
  Avg Overhead/case ((pred - opt) / opt)    = 31.12%


Saving full predictions to './ensemble_results/RF_LR_elasticnet/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 49 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
    

#### Re-evalute on entire dataset

In [197]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR_elasticnet/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR_elasticnet/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 83.95%

Precision by Algorithm:
  UMKP: 85.71% (42 predictions)
  AltRB: 72.39% (134 predictions)
  kPEX: 89.95% (219 predictions)
  Pivot_Br: 85.38% (171 predictions)
  S_Br: 83.51% (194 predictions)

Recall by Algorithm:
  UMKP: 61.00% (61/100 cases)
  AltRB: 94.83% (110/116 cases)
  kPEX: 87.05% (316/363 cases)
  Pivot_Br: 90.27% (204/226 cases)
  S_Br: 91.80% (235/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 74151.60
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.109
  Avg Overhead/case ((pred - opt) / opt)    = 64.02%


Saving full predictions to './ensemble_results/RF_LR_elasticnet/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 122 wrong predictions with time metrics

Top 5 heaviest wrong pre

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [198]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
425,hamming6-2,6,64,1824,28.5,57,57,57,1,64.0,64,64,0.0,0.0,19.9,0.0,6.7,6.8,11.2


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [199]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [200]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [201]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")




Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [202]:
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42 
)

rf_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

clf_elasticnet_strat_y = TorchLogisticClassifier(
        penalty='elasticnet',
        alpha=1e-3,
        beta=0.7,
        lr=1e-2,
        batch_size=128, # batch size = X.shape
        epochs=30,
        temperature=2.0
    )
clf_elasticnet_strat_y.fit(X_train, y_train)

TorchLogisticClassifier(batch_size=128, beta=0.7, epochs=30,
                        penalty='elasticnet', temperature=2.0)

In [203]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y), ('lr02', clf_elasticnet_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=200)),
                             ('lr02',
                              TorchLogisticClassifier(batch_size=128, beta=0.7,
                                                      epochs=30,
                                                      penalty='elasticnet',
                                                      temperature=2.0))])

#### Inference on 20% dataset

In [204]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR_elasticnet/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR_elasticnet/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 75.12%

Precision by Algorithm:
  UMKP: 100.00% (4 predictions)
  AltRB: 65.85% (41 predictions)
  kPEX: 81.82% (44 predictions)
  Pivot_Br: 78.00% (50 predictions)
  S_Br: 72.97% (74 predictions)

Recall by Algorithm:
  UMKP: 63.89% (23/36 cases)
  AltRB: 87.80% (36/41 cases)
  kPEX: 83.74% (103/123 cases)
  Pivot_Br: 83.84% (83/99 cases)
  S_Br: 93.20% (96/103 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 17502.80
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.413
  Avg Overhead/case ((pred - opt) / opt)    = 3486.97%


Saving full predictions to './ensemble_results/RF_LR_elasticnet/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 53 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset  k_value      weight predict

#### Re-evalute on entire dataset

In [205]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR_elasticnet/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR_elasticnet/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 81.84%

Precision by Algorithm:
  UMKP: 100.00% (21 predictions)
  AltRB: 70.15% (134 predictions)
  kPEX: 95.77% (189 predictions)
  Pivot_Br: 86.71% (173 predictions)
  S_Br: 72.43% (243 predictions)

Recall by Algorithm:
  UMKP: 46.00% (46/100 cases)
  AltRB: 91.38% (106/116 cases)
  kPEX: 84.02% (305/363 cases)
  Pivot_Br: 90.27% (204/226 cases)
  S_Br: 93.36% (239/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 84990.40
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.271
  Avg Overhead/case ((pred - opt) / opt)    = 1355.25%


Saving full predictions to './ensemble_results/RF_LR_elasticnet/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 138 wrong predictions with time metrics

Top 5 heaviest wrong predi

### **Balanced train-test split based on Y and K**

In [206]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [207]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")




Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [208]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42  best
)

rf_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate and train
# Instantiate and train
clf_elasticnet_strat_y_K = TorchLogisticClassifier(
    penalty='elasticnet', # 'l1','l2', or 'elasticnet'
    alpha=1e-3,           # regularization strength
    beta=0.7,         # regularization strength of L1 in usint elasticnet
    lr=0.75,              # Learning rate
    batch_size=X_train.shape[0],       # Feeding whole data at a time.
    epochs=100,
    temperature=1.0
)
clf_elasticnet_strat_y_K.fit(X_train, y_train)



TorchLogisticClassifier(batch_size=848, beta=0.7, epochs=100, lr=0.75,
                        penalty='elasticnet')

In [209]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y_K), ('xgb', clf_elasticnet_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=18,
                                                     n_estimators=400)),
                             ('xgb',
                              TorchLogisticClassifier(batch_size=848, beta=0.7,
                                                      epochs=100, lr=0.75,
                                                      penalty='elasticnet'))])

#### Inference on 20% dataset

In [210]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR_elasticnet/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR_elasticnet/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 75.59%

Precision by Algorithm:
  UMKP: 66.67% (12 predictions)
  AltRB: 67.50% (40 predictions)
  kPEX: 77.05% (61 predictions)
  Pivot_Br: 75.51% (49 predictions)
  S_Br: 82.35% (51 predictions)

Recall by Algorithm:
  UMKP: 66.67% (24/36 cases)
  AltRB: 87.80% (36/41 cases)
  kPEX: 87.80% (108/123 cases)
  Pivot_Br: 87.88% (87/99 cases)
  S_Br: 86.41% (89/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 13958.20
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.127
  Avg Overhead/case ((pred - opt) / opt)    = 33.17%


Saving full predictions to './ensemble_results/RF_LR_elasticnet/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 52 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
       

#### Re-evalute on entire dataset

In [211]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_LR_elasticnet/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_LR_elasticnet/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 86.18%

Precision by Algorithm:
  UMKP: 87.23% (47 predictions)
  AltRB: 71.32% (129 predictions)
  kPEX: 93.58% (218 predictions)
  Pivot_Br: 89.88% (168 predictions)
  S_Br: 84.34% (198 predictions)

Recall by Algorithm:
  UMKP: 60.00% (60/100 cases)
  AltRB: 93.97% (109/116 cases)
  kPEX: 90.36% (328/363 cases)
  Pivot_Br: 92.48% (209/226 cases)
  S_Br: 91.80% (235/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 73279.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.096
  Avg Overhead/case ((pred - opt) / opt)    = 64.14%


Saving full predictions to './ensemble_results/RF_LR_elasticnet/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 105 wrong predictions with time metrics

Top 5 heaviest wrong predic

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

#  RF and NN

Hard‐Voting Ensemble model



### Model

In [134]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.metrics import classification_report

In [135]:
# --- 1) Soft Cross‐Entropy with Temperature --------------------------

class SoftCrossEntropyLoss(nn.Module):
    """
    Cross‐entropy loss where logits are divided by a temperature T before softmax.
    """
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # scale logits
        scaled = logits / self.temperature
        return F.cross_entropy(scaled, targets)

In [136]:
# kplex_dev.py

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

# ——— 1) Soft Cross‐Entropy with Temperature ——————————————

class SoftCrossEntropyLoss(nn.Module):
    """
    Cross‐entropy loss where logits are divided by temperature T before softmax.
    """
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        scaled = logits / self.temperature
        return F.cross_entropy(scaled, targets)


# ——— 2) sklearn‐style PyTorch MLP Classifier ——————————————————

class TorchNNClassifier(BaseEstimator, ClassifierMixin):
    """
    A sklearn‐compatible PyTorch MLP with 3–5 hidden layers, soft‐CE loss.
    """
    def __init__(
        self,
        hidden_layers=(128, 64, 32),  # tuple of 3–5 ints
        activation: str = 'relu',     # 'relu' | 'tanh'
        lr: float = 1e-2,
        batch_size: int = 64,
        epochs: int = 50,
        temperature: float = 1.0,
        device: str = None,
        random_state: int = 42
    ):
        self.hidden_layers = hidden_layers
        self.activation    = activation
        self.lr            = lr
        self.batch_size    = batch_size
        self.epochs        = epochs
        self.temperature   = temperature
        self.device        = device
        self.random_state  = random_state

    def _build_model(self, n_features, n_classes):
        layers = []
        in_dim = n_features
        act_fn = nn.ReLU if self.activation=='relu' else nn.Tanh
        # hidden layers
        for h in self.hidden_layers:
            layers.append(nn.Linear(in_dim, h))
            layers.append(act_fn())
            in_dim = h
        # output layer
        layers.append(nn.Linear(in_dim, n_classes))
        return nn.Sequential(*layers)

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        # reproducibility & device
        torch.manual_seed(self.random_state) # comment for checking randomness of algorithm
        self.device_ = torch.device(self.device or ('cuda' if torch.cuda.is_available() else 'cpu'))

        # scale features
        self.scaler_ = StandardScaler().fit(X)
        X_scaled   = self.scaler_.transform(X)

        # map classes to indices
        self.class_to_idx_ = {c:i for i,c in enumerate(self.classes_)}
        y_idx = np.vectorize(self.class_to_idx_.get)(y)

        # DataLoader
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        y_tensor = torch.tensor(y_idx,    dtype=torch.long,   device=self.device_)
        ds = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True)

        # model, loss, optimizer
        self.model_ = self._build_model(n_features, n_classes).to(self.device_)
        criterion = SoftCrossEntropyLoss(self.temperature)
        
        optimizer = optim.SGD(self.model_.parameters(), lr=self.lr, weight_decay=0)

        # training
        for epoch in range(1, self.epochs+1):
            self.model_.train()
            for xb, yb in loader:
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss   = criterion(logits, yb)
                loss.backward()
                optimizer.step()

        return self

    def predict(self, X):
        check_is_fitted(self, ['model_','scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            preds  = logits.argmax(dim=1).cpu().numpy()
        inv_map = {i:c for c,i in self.class_to_idx_.items()}
        return np.vectorize(inv_map.get)(preds)

    def predict_proba(self, X):
        check_is_fitted(self, ['model_','scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            scaled = logits / self.temperature
            probs = F.softmax(scaled, dim=1).cpu().numpy()
        return probs


## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [137]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


Train each base model with sample weights

In [138]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)

# Instantiate and train
# Instantiate and train
clf_nn_strat_y = TorchNNClassifier(
    hidden_layers=(128, 64, 32),
    activation='relu',
    lr=0.15,
    batch_size=32,
    epochs=1000,
    temperature=1,
)
clf_nn_strat_y.fit(X_train, y_train)





TorchNNClassifier(batch_size=32, epochs=1000, lr=0.15, temperature=1)

Build a hard‐voting ensemble and fitting it

In [139]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y = VotingClassifier(
    estimators=[('rf', rf_strat_y), ('nn', clf_nn_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=400)),
                             ('nn',
                              TorchNNClassifier(batch_size=32, epochs=1000,
                                                lr=0.15, temperature=1))])

#### Inference on 20% dataset

In [141]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_NN/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_NN/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 76.53%

Precision by Algorithm:
  UMKP: 83.33% (12 predictions)
  AltRB: 67.44% (43 predictions)
  kPEX: 86.00% (50 predictions)
  Pivot_Br: 70.49% (61 predictions)
  S_Br: 80.85% (47 predictions)

Recall by Algorithm:
  UMKP: 71.05% (27/38 cases)
  AltRB: 95.00% (38/40 cases)
  kPEX: 82.54% (104/126 cases)
  Pivot_Br: 90.91% (90/99 cases)
  S_Br: 91.09% (92/101 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 29115.00
  Total Optimal Time   = 20524.50
  Time Ratio (pred/opt)= 1.419
  Avg Overhead/case ((pred - opt) / opt)    = 172.69%


Saving full predictions to './ensemble_results/RF_NN/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 50 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
           datas

#### Re-evalute on entire dataset

In [143]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_NN/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_NN/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 87.63%

Precision by Algorithm:
  UMKP: 96.36% (55 predictions)
  AltRB: 76.92% (130 predictions)
  kPEX: 96.48% (199 predictions)
  Pivot_Br: 79.82% (218 predictions)
  S_Br: 93.04% (158 predictions)

Recall by Algorithm:
  UMKP: 78.00% (78/100 cases)
  AltRB: 98.28% (114/116 cases)
  kPEX: 87.88% (319/363 cases)
  Pivot_Br: 95.58% (216/226 cases)
  S_Br: 91.80% (235/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 79667.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.191
  Avg Overhead/case ((pred - opt) / opt)    = 59.25%


Saving full predictions to './ensemble_results/RF_NN/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 94 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
    

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on both Y and k**

In [144]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2)

# Label‐encode the string targets into integers 0…4 for XGBoost ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")


Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [145]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
algorithms = ['UMKP', 'AltRB', 'kPEX', 'Pivot_Br', 'S_Br']

# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

# Instantiate and train
clf_nn_strat_y_K = TorchNNClassifier(
    hidden_layers=(128, 64, 32),
    activation='relu',
    lr=0.15,
    batch_size=32,
    # batch_size=X_train.shape[0], 
    epochs=1000,
    temperature=1.0
)

clf_nn_strat_y_K.fit(X_train, y_train)


TorchNNClassifier(batch_size=32, epochs=1000, lr=0.15)

Build a soft‐voting ensemble and fitting it

In [146]:
from sklearn.ensemble import VotingClassifier

ensemble_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_strat_y_K), ('lr0', clf_nn_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=10,
                                                     n_estimators=400)),
                             ('lr0',
                              TorchNNClassifier(batch_size=32, epochs=1000,
                                                lr=0.15))])

#### Inference on 20% dataset

In [147]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_NN/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_NN/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 75.12%

Precision by Algorithm:
  UMKP: 60.00% (5 predictions)
  AltRB: 70.73% (41 predictions)
  kPEX: 80.43% (46 predictions)
  Pivot_Br: 67.12% (73 predictions)
  S_Br: 87.50% (48 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 92.68% (38/41 cases)
  kPEX: 82.93% (102/123 cases)
  Pivot_Br: 90.91% (90/99 cases)
  S_Br: 88.35% (91/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 15246.20
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.231
  Avg Overhead/case ((pred - opt) / opt)    = 27.82%


Saving full predictions to './ensemble_results/RF_NN/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 53 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                

#### Re-evalute on entire dataset

In [148]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_NN/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_NN/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 88.29%

Precision by Algorithm:
  UMKP: 96.49% (57 predictions)
  AltRB: 86.73% (113 predictions)
  kPEX: 94.15% (205 predictions)
  Pivot_Br: 78.64% (206 predictions)
  S_Br: 91.06% (179 predictions)

Recall by Algorithm:
  UMKP: 84.00% (84/100 cases)
  AltRB: 95.69% (111/116 cases)
  kPEX: 87.60% (318/363 cases)
  Pivot_Br: 94.69% (214/226 cases)
  S_Br: 93.75% (240/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 70592.10
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.055
  Avg Overhead/case ((pred - opt) / opt)    = 9.75%


Saving full predictions to './ensemble_results/RF_NN/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 89 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
   

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [149]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
425,hamming6-2,6,64,1824,28.5,57,57,57,1,64.0,64,64,0.0,0.0,19.9,0.0,6.7,6.8,11.2


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [150]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [151]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [152]:
from sklearn.preprocessing import LabelEncoder

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")




Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [153]:
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42 
)

rf_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)

# Instantiate & train NN with 3 layers
clf_nn_holdout_strat_y = TorchNNClassifier(
    hidden_layers=(128, 64, 32),
    activation='relu',
    lr=0.05,
    batch_size=32,
    epochs=1000,
    temperature=1.0
)

clf_nn_holdout_strat_y.fit(X_train_, y_train_)

TorchNNClassifier(batch_size=32, epochs=1000, lr=0.05)

In [154]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y), ('lr0', clf_nn_holdout_strat_y)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=15,
                                                     n_estimators=200)),
                             ('lr0',
                              TorchNNClassifier(batch_size=32, epochs=1000,
                                                lr=0.05))])

#### Inference on 20% dataset

In [155]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_NN/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_NN/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 74.18%

Precision by Algorithm:
  UMKP: 83.33% (6 predictions)
  AltRB: 69.57% (46 predictions)
  kPEX: 79.55% (44 predictions)
  Pivot_Br: 72.22% (72 predictions)
  S_Br: 75.56% (45 predictions)

Recall by Algorithm:
  UMKP: 72.22% (26/36 cases)
  AltRB: 95.12% (39/41 cases)
  kPEX: 84.55% (104/123 cases)
  Pivot_Br: 87.88% (87/99 cases)
  S_Br: 85.44% (88/103 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 18495.60
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.493
  Avg Overhead/case ((pred - opt) / opt)    = 3434.71%


Saving full predictions to './ensemble_results/RF_NN/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 55 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                         dataset  k_value      weight predicted  \
684   

#### Re-evalute on entire dataset

In [156]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_NN/split_and_expand',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_NN/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 85.00%

Precision by Algorithm:
  UMKP: 97.78% (45 predictions)
  AltRB: 79.10% (134 predictions)
  kPEX: 95.41% (196 predictions)
  Pivot_Br: 75.23% (222 predictions)
  S_Br: 87.12% (163 predictions)

Recall by Algorithm:
  UMKP: 74.00% (74/100 cases)
  AltRB: 97.41% (113/116 cases)
  kPEX: 86.23% (313/363 cases)
  Pivot_Br: 92.92% (210/226 cases)
  S_Br: 88.28% (226/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 74265.50
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.110
  Avg Overhead/case ((pred - opt) / opt)    = 92.12%


Saving full predictions to './ensemble_results/RF_NN/split_and_expand\all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 114 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
      

### **Balanced train-test split based on Y and K**

In [157]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [158]:
from sklearn.preprocessing import StandardScaler

# Expanding rows withs ties, multi-label rows

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target'
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")




Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [159]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=18,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42  best
)

rf_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)


# Instantiate & train NN with 3 layers
clf_nn_holdout_strat_y_K = TorchNNClassifier(
    hidden_layers=(128, 64, 32),
    # hidden_layers=(8, 4),
    activation='relu',
    lr=0.01,
    batch_size=32,
    epochs=1000,
    temperature=1.0
)

clf_nn_holdout_strat_y_K.fit(X_train_, y_train_)



TorchNNClassifier(batch_size=32, epochs=1000)

In [160]:
from sklearn.ensemble import VotingClassifier

ensemble_holdout_strat_y_K = VotingClassifier(
    estimators=[('rf', rf_holdout_strat_y_K), ('xgb', clf_l1_strat_y_K)],
    voting='hard',  # averages predicted probabilities else ‘hard’, uses predicted class labels for majority rule voting. 
    # weights=[2,1]  # make RF count twice as much
)

# 3) Fit ensemble (will forward sample_weight to each)
ensemble_holdout_strat_y_K.fit(X_train, y_train)


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(class_weight='balanced',
                                                     max_depth=18,
                                                     n_estimators=400)),
                             ('xgb',
                              TorchLogisticClassifier(batch_size=128,
                                                      epochs=100, lr=0.3,
                                                      penalty='l1'))])

#### Inference on 20% dataset

In [161]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_NN/split_and_expand',
    file_suffix= "inference_test_set_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_NN/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 77.00%

Precision by Algorithm:
  UMKP: 66.67% (12 predictions)
  AltRB: 69.05% (42 predictions)
  kPEX: 77.97% (59 predictions)
  Pivot_Br: 74.55% (55 predictions)
  S_Br: 88.89% (45 predictions)

Recall by Algorithm:
  UMKP: 69.44% (25/36 cases)
  AltRB: 92.68% (38/41 cases)
  kPEX: 86.99% (107/123 cases)
  Pivot_Br: 88.89% (88/99 cases)
  S_Br: 87.38% (90/103 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 12817.60
  Total Optimal Time   = 12389.50
  Time Ratio (pred/opt)= 1.035
  Avg Overhead/case ((pred - opt) / opt)    = 12.66%


Saving full predictions to './ensemble_results/RF_NN/split_and_expand\all_predictions_inference_test_set_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 49 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                  

#### Re-evalute on entire dataset

In [162]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=ensemble_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./ensemble_results/RF_NN/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./ensemble_results/RF_NN/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 86.05%

Precision by Algorithm:
  UMKP: 83.33% (36 predictions)
  AltRB: 66.22% (148 predictions)
  kPEX: 94.14% (222 predictions)
  Pivot_Br: 90.53% (169 predictions)
  S_Br: 88.65% (185 predictions)

Recall by Algorithm:
  UMKP: 53.00% (53/100 cases)
  AltRB: 97.41% (113/116 cases)
  kPEX: 90.91% (330/363 cases)
  Pivot_Br: 92.92% (210/226 cases)
  S_Br: 92.97% (238/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 70714.60
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.057
  Avg Overhead/case ((pred - opt) / opt)    = 13.17%


Saving full predictions to './ensemble_results/RF_NN/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 106 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
    

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br